# FinS4 Build Corpus — S4 Chunks + Training Pairs

## Goal
1. Build S4 (token-exact) chunks from all 41 PDFs across 4 categories
2. Map ~2,400 Groq questions from `intent_dataset.csv` → nearest source S4 chunk
3. Save `s4_corpus.csv` and `training_pairs.csv` for fine-tuning

## Why S4?
- Token-exact (200 tokens/50 overlap) — 0% truncation vs word-based chunks
- Same strategy that won chunk evaluation (Recall@10 = 0.80)

## Output
| File | Purpose |
|------|---------|
| `fine_tuning/s4_corpus.csv` | All S4 chunks with chunk_id, category, source_file, text |
| `fine_tuning/training_pairs.csv` | (question, chunk_text, chunk_id, category) pairs for fine-tuning |

In [ ]:
import sys, os
from pathlib import Path

_root = Path().resolve()
for _ in range(5):
    if (_root / 'config.py').exists():
        break
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root))

from config import (CONSUMER_PROTECTION_PATH, PAYMENT_INDUSTRY_PATH,
                    REGULATORY_PATH, COMPLAINT_PROCEDURES_PATH)

FINE_TUNE_DIR   = 'fine_tuning'
DATASET_PATH    = os.path.join('intent_classification', 'intent_dataset.csv')
S4_CORPUS_PATH  = os.path.join(FINE_TUNE_DIR, 's4_corpus.csv')
PAIRS_PATH      = os.path.join(FINE_TUNE_DIR, 'training_pairs.csv')
MINILM_NAME     = 'sentence-transformers/all-MiniLM-L6-v2'

# All 4 category → folder mappings
CATEGORY_DIRS = {
    'Regulatory':          REGULATORY_PATH,
    'Consumer_Protection': CONSUMER_PROTECTION_PATH,
    'Payment_Industry':    PAYMENT_INDUSTRY_PATH,
    'Synthetic_Policies':  COMPLAINT_PROCEDURES_PATH,
}

os.makedirs(FINE_TUNE_DIR, exist_ok=True)
print('Repo root      :', _root)
for cat, path in CATEGORY_DIRS.items():
    pdfs = list(Path(path).glob('*.pdf'))
    print('  [{:<22}]  {} PDFs  →  {}'.format(cat, len(pdfs), path))

In [ ]:
import sys
!{sys.executable} -m pip install -q pdfplumber transformers sentence-transformers faiss-cpu pandas tqdm

In [ ]:
import re
import numpy as np
import pandas as pd
import faiss
import pdfplumber
from tqdm import tqdm
from pathlib import Path
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer

print('Imports OK')

## Cell 3 — Build S4 Chunks from All PDFs
Extract all PDFs → token-exact chunks (200 tokens / 50 overlap) → save `s4_corpus.csv`

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MINILM_NAME)

def extract_pdf_text(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            raw = page.extract_text()
            if raw:
                clean = re.sub(r'\n{3,}', '\n\n', raw)
                clean = re.sub(r'[ \t]{2,}', ' ', clean)
                clean = re.sub(r'[^\x00-\x7F]+', ' ', clean).strip()
                if len(clean) > 50:
                    pages.append(clean)
    return '\n\n'.join(pages)

def chunk_token_exact(text, tokenizer, target=200, overlap=50):
    tokens = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    step, chunks, start = target - overlap, [], 0
    while start < len(tokens):
        end = min(start + target, len(tokens))
        txt = tokenizer.decode(tokens[start:end], skip_special_tokens=True).strip()
        if len(txt) > 30:
            chunks.append({'text': txt, 'token_count': end - start})
        start += step
        if end == len(tokens):
            break
    return chunks

# Build S4 corpus from all PDFs
all_chunks = []
chunk_counter = {}

for category, folder in CATEGORY_DIRS.items():
    pdfs = sorted(Path(folder).glob('*.pdf'))
    print('[{}]  {} PDFs'.format(category, len(pdfs)))
    chunk_counter[category] = 0

    for pdf_path in pdfs:
        try:
            text = extract_pdf_text(str(pdf_path))
            chunks = chunk_token_exact(text, tokenizer)
            for i, ch in enumerate(chunks):
                # chunk_id: category prefix + file stem + index
                stem = re.sub(r'[^a-z0-9]', '_', pdf_path.stem.lower())[:20]
                chunk_id = 's4_{}_{}_{:04d}'.format(category[:3].lower(), stem, i)
                all_chunks.append({
                    'chunk_id':    chunk_id,
                    'category':    category,
                    'source_file': pdf_path.name,
                    'text':        ch['text'],
                    'token_count': ch['token_count'],
                })
            chunk_counter[category] += len(chunks)
            print('  {} → {} chunks'.format(pdf_path.name[:40], len(chunks)))
        except Exception as e:
            print('  ERROR {}: {}'.format(pdf_path.name, e))

corpus_df = pd.DataFrame(all_chunks)
corpus_df.to_csv(S4_CORPUS_PATH, index=False)

print('\n' + '='*55)
print('  S4 CORPUS SUMMARY')
print('='*55)
print('  Total chunks: {:,}'.format(len(corpus_df)))
for cat, n in chunk_counter.items():
    print('  {:<25} {:>6,} chunks'.format(cat, n))
print('='*55)
print('Saved:', S4_CORPUS_PATH)

## Cell 4 — Map Groq Questions → Nearest S4 Chunk
For each Groq question in `intent_dataset.csv`:
- Filter S4 chunks by same category
- Encode question + category chunks with MiniLM
- Find nearest chunk via cosine similarity → this is the "source chunk"
- Threshold: similarity > 0.3 to keep only confident matches

In [ ]:
# Load Groq questions — keep only PDF-domain rows (word_count >= 8)
intent_df = pd.read_csv(DATASET_PATH)
intent_df['word_count'] = intent_df['text'].str.split().str.len()
groq_df   = intent_df[intent_df['word_count'] >= 8].reset_index(drop=True)

print('Groq questions loaded: {:,}'.format(len(groq_df)))
print(groq_df['category'].value_counts().to_string())

# Load embedding model
print('\nLoading MiniLM for similarity mapping...')
embed_model = SentenceTransformer(MINILM_NAME)

# Build per-category FAISS index from S4 corpus
CATEGORIES = ['Regulatory', 'Consumer_Protection', 'Payment_Industry', 'Synthetic_Policies']
SIM_THRESHOLD = 0.25   # min cosine similarity to keep pair

cat_indexes   = {}   # category → (faiss_index, chunk_ids, chunk_texts)

print('\nBuilding per-category FAISS indexes...')
for cat in CATEGORIES:
    cat_chunks = corpus_df[corpus_df['category'] == cat].reset_index(drop=True)
    texts      = cat_chunks['text'].tolist()
    ids        = cat_chunks['chunk_id'].tolist()

    embs = embed_model.encode(texts, batch_size=128, normalize_embeddings=True,
                               show_progress_bar=False, convert_to_numpy=True)
    idx  = faiss.IndexFlatIP(embs.shape[1])
    idx.add(embs.astype(np.float32))
    cat_indexes[cat] = (idx, ids, texts)
    print('  [{:<22}]  {:,} chunks indexed'.format(cat, len(texts)))

print('\nMapping questions → nearest chunks...')
training_rows = []
skipped = 0

for cat in CATEGORIES:
    cat_questions = groq_df[groq_df['category'] == cat]['text'].tolist()
    if not cat_questions:
        continue

    idx, chunk_ids, chunk_texts = cat_indexes[cat]

    q_embs = embed_model.encode(cat_questions, batch_size=128,
                                normalize_embeddings=True,
                                show_progress_bar=True,
                                convert_to_numpy=True)
    scores, indices = idx.search(q_embs.astype(np.float32), 1)

    for q_text, score, chunk_idx in zip(cat_questions, scores[:, 0], indices[:, 0]):
        if score >= SIM_THRESHOLD:
            training_rows.append({
                'question':        q_text,
                'chunk_text':      chunk_texts[chunk_idx],
                'chunk_id':        chunk_ids[chunk_idx],
                'category':        cat,
                'sim_score':       round(float(score), 4),
            })
        else:
            skipped += 1

pairs_df = pd.DataFrame(training_rows)
pairs_df.to_csv(PAIRS_PATH, index=False)

print('\n' + '='*55)
print('  TRAINING PAIRS SUMMARY')
print('='*55)
print('  Total pairs  : {:,}'.format(len(pairs_df)))
print('  Skipped (low sim): {:,}'.format(skipped))
print('  Avg similarity   : {:.4f}'.format(pairs_df['sim_score'].mean()))
print()
for cat in CATEGORIES:
    n = len(pairs_df[pairs_df['category'] == cat])
    print('  {:<25} {:>6,}'.format(cat, n))
print('='*55)
print('Saved:', PAIRS_PATH)